In [ ]:
%%sql


In [ ]:
# =========================
# REPRODUCTIBILITÉ
# =========================
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Classification Chats vs Chiens - CNN Simple

Ce notebook realise une classification binaire d'images (chat ou chien) en plusieurs etapes. Chaque etape est expliquee en detail pour comprendre ce que fait le code et pourquoi.

## Etape 1 : Import des bibliotheques et configuration

On importe les modules Python necessaires et on definit les parametres globaux. Ces variables determinent le chemin des donnees, la taille des images, et combien d'images charger par classe.

In [ ]:
# Bibliotheques pour le calcul numerique (numpy), l'apprentissage profond (tensorflow/keras),
# les graphiques (matplotlib), et la manipulation de fichiers (pathlib, zipfile, PIL)
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
from PIL import Image

# Configuration : chemin du fichier zip contenant les images
ZIP_PATH = 'kagglecatsanddogs_5340.zip'
# Dossier ou extraire les images
DATA_DIR = 'cats_dogs_data'
# Taille cible des images en pixels (largeur et hauteur). Toutes les images seront redimensionnees
# a 128x128 pour que le modele ait des entrees de taille fixe
IMG_SIZE = 128
# Nombre max d'images a charger par classe (Chat et Chien). Limiter reduit le temps d'entrainement
# mais moins de donnees peut diminuer la precision. Augmenter a 5000+ pour de meilleurs resultats
MAX_IMAGES_PER_CLASS = 2000

## Etape 2 : Extraction et chargement des donnees

On extrait le fichier zip si necessaire, puis on charge les images. Chaque image est convertie en RGB et redimensionnee. Les etiquettes (labels) : 0 pour Chat, 1 pour Chien.

In [ ]:
def load_cats_dogs(zip_path=ZIP_PATH, data_dir=DATA_DIR, img_size=IMG_SIZE, max_per_class=MAX_IMAGES_PER_CLASS):
    """
    Extrait le zip et charge les images de chats et chiens.
    Retourne X (tableau d'images) et y (etiquettes : 0=Chat, 1=Chien).
    """
    # Extraire le zip uniquement si le dossier de donnees n'existe pas encore
    if Path(data_dir).exists() is False:
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(data_dir)
    
    base = Path(data_dir) / 'PetImages'
    X, y = [], []
    for label, folder in enumerate(['Cat', 'Dog']):  # label 0 = Chat, 1 = Chien
        folder_path = base / folder
        if not folder_path.exists():
            continue
        count = 0
        for img_path in sorted(folder_path.glob('*.jpg')):
            if count >= max_per_class:
                break
            try:
                img = Image.open(img_path).convert('RGB').resize((img_size, img_size))
                X.append(np.array(img))  # Conversion PIL -> numpy
                y.append(label)
                count += 1
            except Exception:
                pass
    return np.array(X), np.array(y)

X_raw, y = load_cats_dogs()
print(f"Charge : {X_raw.shape[0]} images, forme {X_raw.shape[1:]}")
print(f"Classes : Chat=0, Chien=1 | Effectifs : Chat={np.sum(y==0)}, Chien={np.sum(y==1)}")

## Etape 3 : Visualisation d'exemples d'images

On affiche 10 images (5 chats, 5 chiens) pour verifier que le chargement s'est bien passe et voir a quoi ressemblent les donnees.

In [ ]:
# Creer une grille 2 lignes x 5 colonnes pour afficher 10 images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    # i % 2 donne 0 ou 1 (alternance Chat/Chien), i // 2 donne l'indice dans la classe
    idx = np.where(y == i % 2)[0][i // 2]
    ax.imshow(X_raw[idx])
    ax.set_title('Chat' if y[idx] == 0 else 'Chien')
    ax.axis('off')
plt.suptitle('Exemples d images', fontsize=12)
plt.tight_layout()
plt.show()

## Etape 4 : Pre-traitement - Normalisation des pixels

Les pixels bruts sont des entiers entre 0 et 255. On les divise par 255 pour obtenir des valeurs entre 0 et 1. Cela stabilise l'entrainement du reseau de neurones : des valeurs trop grandes peuvent faire diverger les poids.

In [ ]:
# Conversion en float et normalisation : valeurs de pixels entre 0 et 1
X = X_raw.astype(np.float32) / 255.0
print(f"Plage des pixels apres normalisation : [{X.min():.2f}, {X.max():.2f}]")

## Etape 5 : Division train / validation / test

On separe les donnees en trois ensembles :
- **Train** : pour entrainer le modele (le modele apprend sur ces images)
- **Validation** : pour suivre la performance pendant l'entrainement et arreter au bon moment (eviter l'overfitting)
- **Test** : pour evaluer le modele a la fin sur des images jamais vues

stratify=y garantit que la proportion Chat/Chien reste la meme dans chaque ensemble. random_state=42 assure la reproductibilite.

In [ ]:
from sklearn.model_selection import train_test_split

# D'abord isoler 15% pour le test (donnees jamais vues jusqu'a la fin)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
# Puis diviser le reste : 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

print(f"Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")

In [ ]:
# =========================
# CONFIGURATION EXPERIENCE
# =========================

EXP_NAME = "Exp12_3000perclass"

LEARNING_RATE = 0.0005
EPOCHS = 20
BATCH_SIZE = 32

PATIENCE_EARLYSTOP = 5
PATIENCE_REDUCE_LR = 2
REDUCE_FACTOR = 0.5

L2_FACTOR = 0.0005
DROPOUT_RATE = 0.4

## Etape 10 : Evaluation sur le jeu de test

On evalue le modele sur des images jamais vues (test set). C'est la mesure finale de la qualite. Un ecart modere entre precision train et test est normal. Un ecart tres grand indique de l'overfitting.

In [ ]:
from tensorflow.keras import layers, models, regularizers

model = models.Sequential([

    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Bloc 1
    layers.Conv2D(32, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(32, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D(2,2),

    # Bloc 2
    layers.Conv2D(64, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(64, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D(2,2),

    # Bloc 3
    layers.Conv2D(128, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(128, (3,3), kernel_regularizer=regularizers.l2(L2_FACTOR)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D(2,2),

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=regularizers.l2(L2_FACTOR)
    ),

    layers.Dropout(DROPOUT_RATE),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE_EARLYSTOP,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=REDUCE_FACTOR,
    patience=PATIENCE_REDUCE_LR,
    min_lr=1e-6,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    f"{EXP_NAME}_best.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=5,
    width_shift_range=0.03,
    height_shift_range=0.03,
    zoom_range=0.03,
    horizontal_flip=True,
    fill_mode="nearest"
)

import time
import math

steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)

start_time = time.time()

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True),
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

training_time = round(time.time() - start_time, 2)
print("Temps d'entraînement :", training_time, "s")

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nPrécision sur le jeu de test : {test_acc*100:.2f}%")

from sklearn.metrics import f1_score

# Prédictions binaires
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

# F1-score
f1 = f1_score(y_test, y_pred, average='macro')

# Nombre d'erreurs
erreurs = int(np.sum(y_pred != y_test))

# Accuracy en %
accuracy_percent = round(test_acc * 100, 2)

# Learning rate actuel (plus fiable que history)
lr_final = float(model.optimizer.learning_rate.numpy())

print(f"F1-score (macro) : {f1:.3f}")
print(f"Erreurs totales : {erreurs}")
print(f"Learning rate final : {lr_final}")

In [ ]:
import tensorflow as tf

best_model = tf.keras.models.load_model(f"{EXP_NAME}_best.keras")

test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test accuracy (BEST checkpoint) : {test_acc*100:.2f}%")

Etape 11
- **Graphique 1 : Matrice de confusion**
- **Graphique 2 : Distribution de confiance**
- **Graphique 3 : Images mal classifiées**
- **Comparaison des expériences (graphique à barres)**

In [ ]:
plt.figure(figsize=(12,5))

# Graphique 1 : Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy pendant l entraînement')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Graphique 2 : Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss pendant l entraînement')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# ANALYSE DU MODELE - VERSION FINALE
# ===============================

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ---- Prédictions ----
y_pred_proba = model.predict(X_test).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

# ---- Rapport de classification ----
print("Rapport de classification :")
print(classification_report(y_test, y_pred, target_names=['Chat', 'Chien']))

# ---- Matrice de confusion ----
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Chat','Chien'],
            yticklabels=['Chat','Chien'])
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title(f"Matrice de confusion - {EXP_NAME}")
plt.tight_layout()
plt.show()

# ---- Distribution des probabilités ----
plt.figure(figsize=(8,5))
plt.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.6, label='Chats')
plt.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.6, label='Chiens')
plt.axvline(0.5, color='red', linestyle='--', label='Seuil 0.5')
plt.legend()
plt.title(f"Distribution des probabilités - {EXP_NAME}")
plt.xlabel("Probabilité prédite (Chien)")
plt.ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

In [ ]:
errors = np.where(y_pred != y_test)[0]
confidence = np.abs(y_pred_proba[errors] - 0.5)

sorted_errors = errors[np.argsort(confidence)[::-1]]

fig, axes = plt.subplots(3, 3, figsize=(8,8))
for i, ax in enumerate(axes.flat):
    idx = sorted_errors[i]
    ax.imshow(X_test[idx])
    ax.set_title(f"Vrai:{y_test[idx]} | Pred:{y_pred[idx]}")
    ax.axis('off')
plt.suptitle("Erreurs les plus confiantes")
plt.show()

## Etape 12 : Créer la structure DataFrame



In [ ]:
import pandas as pd
import os

# Charger ancien fichier CSV s'il existe
if os.path.exists("suivi_experiences.csv"):
    df_suivi = pd.read_csv("suivi_experiences.csv")
else:
    df_suivi = pd.DataFrame()

# Créer nouvelle ligne résultat

nouvelle_ligne = {
    "Expérience": EXP_NAME,
    "Learning Rate": LEARNING_RATE,
    "Epochs": EPOCHS,
    "Accuracy (%)": accuracy_percent,
    "Loss": round(test_loss, 4),
    "F1-score (macro)": round(f1, 3),
    "Erreurs totales": erreurs,
    "Temps (s)": training_time,
    "LR final": lr_final
}
# Ajouter au dataframe
df_suivi = pd.concat([df_suivi, pd.DataFrame([nouvelle_ligne])], ignore_index=True)

# Sauvegarder automatiquement
df_suivi.to_csv("suivi_experiences.csv", index=False)

df_suivi.sort_values(by="Accuracy (%)", ascending=False)

Graphique de comparaison automatique


In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(df_suivi["Expérience"], df_suivi["Accuracy (%)"])

baseline_value = df_suivi["Accuracy (%)"].min()
plt.axhline(baseline_value, color='red', linestyle='--', label='Baseline')

# Afficher valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.5,
             f"{height:.2f}%", ha='center')

plt.title("Comparaison des expériences")
plt.ylabel("Accuracy (%)")
plt.xlabel("Expérience")
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
model.save(f"{EXP_NAME}.keras")
print(f"✅ Modèle sauvegardé : {EXP_NAME}.keras")
print(f"✅ Meilleur modèle sauvegardé : {EXP_NAME}_best.keras (via checkpoint)")

---